# Compound Wind-Hydro Energy Droughts in Patagonia


### Research Question
**Can compound wind hydro energy droughts in Patagonia be predicted from 
large scale climate modes, and if so, how far in advance?**

### Project Overview

This analysis integrates 47 years of climate data (1979–2025) with regional wind and hydroelectric generation potential across Patagonia 
(38°S–47.5°S, 72°W–62°W) to determine whether the Southern Annular Mode (SAM), Oceanic Niño Index (ONI), and Indian Ocean Dipole (IOD) carry predictive skill for compound energy droughts.

**Key regions of focus:**
- **Hydroelectric:** Limay-Neuquén basin dams (Chocón, Piedra del Águila, Alicurá, Futaleufú)
- **Wind:** Atlantic coast wind farms (Rawson, Trelew, Puerto Madryn, Manantiales Behr)
- **Temporal domain:** Monthly aggregation, 1979–2025 (47 years)
- **Compound metric:** WHDI = standardised wind anomaly + standardised runoff anomaly

### Summary of Findings

**Yes, compound droughts are predictable** but the predictive source inverts across the forecast horizon.

**Local conditions dominate short leads (1–2 months):** +68% and +39% skill vs climatology

**Climate modes dominate medium leads (5–6 months):** +4.6% skill, p<0.05 (IOD-driven)

**Lead 3 is a transition cliff edge:** Both sources collapse, forecast skill near zero

**Super El Niño events are hardest to predict:** 1997–98 and 2015–16 show 1.6× larger forecast errors despite extreme climate signals

**Operational implication:** Forecast systems must switch paradigms, use local persistence for months 1–2, climate teleconnections for months 5–6, with no 
single hybrid model dominating both horizons.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json

warnings.filterwarnings("ignore")

# Set style for all visualizations in this notebook
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9
plt.rcParams["legend.fontsize"] = 9
plt.rcParams["lines.linewidth"] = 1.5
plt.rcParams["patch.linewidth"] = 0.5

# Notebook 2
df_whdi = pd.read_csv(
    "../data/processed/whdi_timeseries.csv", index_col="date", parse_dates=True
)
print(f"WHDI Timeseries: {df_whdi.shape}")

df_droughts = pd.read_csv("../results/tables/drought_catalog.csv")
print(f"Drought Catalog: {df_droughts.shape}")

df_climatology = pd.read_csv("../results/tables/monthly_climatology.csv", index_col=0)
print(f"Monthly Climatology: {df_climatology.shape}")

# Notebook 3
df_lag_corr = pd.read_csv("../results/tables/lag_correlations.csv", index_col=0)
print(f"Lag Correlations: {df_lag_corr.shape}")

df_seasonal_lag = pd.read_csv(
    "../results/tables/seasonal_lag_correlations.csv", index_col=0
)
print(f"Seasonal Lag Correlations: {df_seasonal_lag.shape}")

df_trends = pd.read_csv("../results/tables/trend_results.csv")
print(f"Trend Results: {df_trends.shape}")

# Notebook 4
df_performance = pd.read_csv("../results/tables/model_performance.csv")
print(f"Model Performances: {df_performance.shape}")

df_skills = pd.read_csv("../results/tables/skill_scores.csv")
print(f"Skill Scores: {df_skills.shape}")

df_significance = pd.read_csv("../results/tables/significance_tests.csv")
print(f"Significance Tests: {df_significance.shape}")

df_anomalies = pd.read_csv("../results/tables/anomalous_months_summary.csv")
print(f"Anomalous Months: {df_anomalies.shape}")

with open("../results/tables/feature_decisions.json", "r") as f:
    feature_decisions = json.load(f)
print(f"Feature decisions: {len(feature_decisions)} parameters")

WHDI Timeseries: (564, 13)
Drought Catalog: (31, 7)
Monthly Climatology: (12, 16)
Lag Correlations: (195, 8)
Seasonal Lag Correlations: (468, 7)
Trend Results: (5, 6)
Model Performances: (3456, 9)
Skill Scores: (2304, 8)
Significance Tests: (288, 11)
Anomalous Months: (28, 7)
Feature decisions: 4 parameters


In [2]:
# Study Domain
STUDY_BOUNDS = [-72.0, -47.5, -62.0, -38.0]
STUDY_CENTRE = [-42.5, -67.5]

# Drought Thresholds
DROUGHT_THRESHOLDS = [-1.0, -1.5, -2.0]

# Lead Times Analysed
LEAD_TIMES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
PRIMARY_TARGET = "whdi_3"
SECONDARY_TARGET = "whdi_6"

# Models and ML things
MODELS = [
    "elasticnet",
    "random_forest",
    "xgboost",
    "lstm",
    "climatology",
    "persistence",
]
ML_MODELS = ["elasticnet", "random_forest", "xgboost", "lstm"]
FEATURE_GROUPS = ["climate_only", "local_only", "combined"]
CLIMATE_MODES = ["SAM", "ONI", "IOD"]

# Maintain consistency across all plots etc
COLOURS = {
    "elasticnet": "#0095ff",
    "random_forest": "#ff0000",
    "xgboost": "#ff9900",
    "lstm": "#00ff2a",
    "climatology": "black",
    "persistence": "lightgrey",
    "climate_only": "#E91E63",
    "local_only": "#2196F3",
    "combined": "#4CAF50",
}

print("Parameters set:")
print(f"  Study region: {STUDY_BOUNDS}")
print(f"  Lead times: {LEAD_TIMES}")
print(f"  Primary target: {PRIMARY_TARGET}")
print(f"  Feature groups: {FEATURE_GROUPS}")
print(f"  ML models: {ML_MODELS}")

Parameters set:
  Study region: [-72.0, -47.5, -62.0, -38.0]
  Lead times: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  Primary target: whdi_3
  Feature groups: ['climate_only', 'local_only', 'combined']
  ML models: ['elasticnet', 'random_forest', 'xgboost', 'lstm']


## Utility Functions

In [ ]:
def get_best_model_at_lead(df_skills_subset, target, lead, feature_group):
    """
    Return the best performing model (by skill vs climatology) at a given lead.
    """
    subset = df_skills_subset[
        (df_skills_subset["target"] == target)
        & (df_skills_subset["lead"] == lead)
        & (df_skills_subset["feature_group"] == feature_group)
    ]
    if len(subset) == 0:
        return None
    return subset.groupby("model")["skill_vs_clim"].mean().idxmax()


def summarise_skill_at_lead(df_skills_subset, target, lead):
    """
    Summarise skill across feature groups at a single lead time.
    Returns dict with climate_only, local_only, combined mean skills.
    """
    best_model = get_best_model_at_lead(df_skills_subset, target, lead, "combined")
    if best_model is None:
        return None

    result = {}
    for group in FEATURE_GROUPS:
        skill = df_skills_subset[
            (df_skills_subset["target"] == target)
            & (df_skills_subset["lead"] == lead)
            & (df_skills_subset["feature_group"] == group)
            & (df_skills_subset["model"] == best_model)
        ]["skill_vs_clim"].mean()
        result[group] = skill

    return result


def count_drought_events(whdi_ts, threshold=DROUGHT_THRESHOLDS[0]):
    """
    Count contiguous drought events in WHDI timeseries.
    Returns list of (start_date, end_date, duration, peak_severity).
    """
    in_drought = whdi_ts < threshold
    events = []

    start = None
    for date, is_drought in in_drought.items():
        if is_drought and start is None:
            start = date
        elif not is_drought and start is not None:
            end = date
            duration = (end - start).days // 30
            peak = whdi_ts[start:end].min()
            events.append((start, end, duration, peak))
            start = None

    return events

✓ Utility functions defined
  - get_best_model_at_lead()
  - summarise_skill_at_lead()
  - count_drought_events()
